In [21]:
import pandas as pd
import numpy as np
import re
import nltk

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix

In [22]:
train_df = pd.read_csv(
    "train_data.txt",
    sep=" ::: ",
    names=["ID","TITLE","GENRE","DESCRIPTION"],
    engine="python"
)

test_df = pd.read_csv(
    "test_data.txt",
    sep=" ::: ",
    names=["ID","TITLE","DESCRIPTION"],
    engine="python"
)

solution_df = pd.read_csv(
    "test_data_solution.txt",
    sep=" ::: ",
    names=["ID","TITLE","GENRE","DESCRIPTION"],
    engine="python"
)

In [39]:
print("Train Shape:", train_df.shape)
print("Test Shape:", test_df.shape)
print("Solution Shape:", solution_df.shape)

Train Shape: (54214, 5)
Test Shape: (54200, 4)
Solution Shape: (54200, 5)


In [24]:
# checking missing values
print(train_df.isnull().sum())

ID             0
TITLE          0
GENRE          0
DESCRIPTION    0
dtype: int64


In [25]:
# No of genres and genre distribution
print("Total Genres:", train_df["GENRE"].nunique()) 
train_df["GENRE"].value_counts().head(15)

Total Genres: 27


GENRE
drama          13613
documentary    13096
comedy          7447
short           5073
horror          2204
thriller        1591
action          1315
western         1032
reality-tv       884
family           784
adventure        775
music            731
romance          672
sci-fi           647
adult            590
Name: count, dtype: int64

In [26]:
# text cleaning
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    words = text.split()
    words = [word for word in words if word not in stop_words]
    words = [lemmatizer.lemmatize(word) for word in words]
    return " ".join(words)

In [27]:
train_df["clean_desc"] = train_df["DESCRIPTION"].apply(clean_text)
test_df["clean_desc"] = test_df["DESCRIPTION"].apply(clean_text)
solution_df["clean_desc"] = solution_df["DESCRIPTION"].apply(clean_text)

In [28]:
X_train = train_df["clean_desc"]
y_train = train_df["GENRE"]
X_test = test_df["clean_desc"]
y_test = solution_df["GENRE"]

In [29]:
model = Pipeline([
    (
        "tfidf",
        TfidfVectorizer(
            max_features=5000,
            ngram_range=(1,2)
        )
    ),
    (
        "classifier",
        LogisticRegression(
            max_iter=1000
        )
    )
])

In [30]:
model.fit(X_train, y_train)
predictions = model.predict(X_test)

In [31]:
accuracy = accuracy_score(y_test, predictions)
print("Accuracy:", round(accuracy*100,2), "%")

Accuracy: 58.63 %


In [32]:
print(classification_report(y_test, predictions))

C:\Users\amolk\anaconda3\envs\tensorflow\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
C:\Users\amolk\anaconda3\envs\tensorflow\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


              precision    recall  f1-score   support

      action       0.46      0.29      0.36      1314
       adult       0.62      0.25      0.36       590
   adventure       0.57      0.17      0.26       775
   animation       0.49      0.07      0.13       498
   biography       0.00      0.00      0.00       264
      comedy       0.53      0.59      0.56      7446
       crime       0.33      0.04      0.07       505
 documentary       0.67      0.85      0.75     13096
       drama       0.55      0.77      0.64     13612
      family       0.49      0.10      0.16       783
     fantasy       0.54      0.06      0.11       322
   game-show       0.85      0.54      0.66       193
     history       0.00      0.00      0.00       243
      horror       0.65      0.58      0.61      2204
       music       0.64      0.44      0.52       731
     musical       0.17      0.01      0.03       276
     mystery       0.33      0.01      0.02       318
        news       0.67    

C:\Users\amolk\anaconda3\envs\tensorflow\Lib\site-packages\sklearn\metrics\_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [33]:
cm = confusion_matrix(y_test, predictions)
print(cm)

[[  383     2    11     1     0   143     7   132   445     0     2     0
      0    37     2     2     1     0     2     1    26    52    11     1
     41     0    12]
 [    6   148    23     0     0   167     0    34   146     0     0     0
      0    11     1     0     0     0     1     1     1    48     2     0
      1     0     0]
 [   42    40   130     9     0   108     0   120   209     5     3     0
      0    28     0     0     1     0     8     0    13    38     2     0
     11     0     8]
 [   30     0    14    36     0   121     0    83    88    16     6     1
      0    20     2     0     0     0     3     0    21    53     2     0
      2     0     0]
 [    0     0     0     0     0    17     0   176    57     0     0     0
      0     2     3     1     0     0     0     0     0     7     1     0
      0     0     0]
 [   63     8     6     4     0  4395     4   423  2075    16     1     5
      0    85    13     1     1     2    20    11    17   248     3    10
     17

In [34]:
results = pd.DataFrame({
    "Movie_ID": test_df["ID"],
    "Predicted_Genre": predictions
})

results.head()

,Movie_ID,Predicted_Genre
0,1,drama
1,2,drama
2,3,documentary
3,4,drama
4,5,drama


In [35]:
results.to_csv(
    "movie_genre_predictions.csv",
    index=False
)

print("Predictions Saved Successfully")

Predictions Saved Successfully


In [38]:
sample = """
An unlikely group of heroes bands together
to save their kingdom from invading monsters.
"""

sample = clean_text(sample)

prediction = model.predict([sample])

print("Predicted Genre:", prediction[0])

Predicted Genre: animation


In [40]:
pip freeze > requirements.txt

Note: you may need to restart the kernel to use updated packages.
